# Análisis Geoespacial IPM - Zona de Intervención: Roosevelt

Este notebook prioriza la visualización cartográfica de las 5 variables más relevantes del Índice de Pobreza Multidimensional (IPM) en el corredor Roosevelt (Buffer 100m).

### Contexto y Cifras Generales (Corte Mayo 2026)

Para la interpretación de los resultados, se deben considerar dos lógicas opuestas:
1. **Índice de Condición Social (ICS):** Es un indicador directo; un valor del **100% representa el escenario óptimo** (bienestar y acceso pleno).
2. **Índice de Pobreza Multidimensional (IPM):** Es un indicador inverso; un valor del **100% representa el escenario crítico** (pobreza absoluta).

A nivel distrital, Cali presenta una marcada brecha territorial. La **severidad de la pobreza (IPM promedio en manzanas con incidencia > 0)** es del **15.28% en el área urbana**, mientras que en el **área rural se dispara al 38.33%**, impulsada principalmente por carencias en infraestructura básica en los corregimientos.

Este análisis se enfoca en el corredor Roosevelt para identificar las privaciones específicas que afectan a este territorio de intervención.

In [ ]:
# 1. Instalación de dependencias (solo en Colab)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas matplotlib seaborn openpyxl -q

In [ ]:
# 2. Importar librerías
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from matplotlib.patheffects import withStroke
from matplotlib.colors import BoundaryNorm
import matplotlib.patches as mpatches

sns.set_style('white')
print('Librerías listas')

In [ ]:
# 3. Definir rutas con detección de entorno
REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !cd {REPO_DIR} && git pull
    BASE_DIR = REPO_DIR
else:
    if os.path.exists('indice_Pobreza'): BASE_DIR = '.'
    elif os.path.exists('../indice_Pobreza'): BASE_DIR = '..'
    elif os.path.exists('../../indice_Pobreza'): BASE_DIR = '../..'
    else: BASE_DIR = '.'

DATA_BASE = os.path.join(BASE_DIR, 'indice_Pobreza/data')

# Capas
PATH_IPM_VARS_FULL = os.path.join(DATA_BASE, 'geojson_ipm/Mzn_ipm_variables.geojson')
PATH_MANZANAS_FONDO = os.path.join(DATA_BASE, 'geojson_Manzanas_catastrales/geojson_Manzanas_catastrales.geojson')
PATH_AREA_ESTUDIO = os.path.join(DATA_BASE, 'geojson_poligonos_territorio_ITT/poligono_Roosevelt_Buffer_100.geojson')

# Cargar datos
gdf_full = gpd.read_file(PATH_IPM_VARS_FULL)
gdf_fondo = gpd.read_file(PATH_MANZANAS_FONDO)
gdf_area = gpd.read_file(PATH_AREA_ESTUDIO)

# Unificar CRS a WGS84
gdf_full = gdf_full.to_crs('EPSG:4326')
gdf_fondo = gdf_fondo.to_crs('EPSG:4326')
gdf_area = gdf_area.to_crs('EPSG:4326')

# Filtrado espacial preciso (Centroide dentro del Buffer)
gdf_p = gdf_full.to_crs("EPSG:3115")
area_p = gdf_area.to_crs("EPSG:3115")
idx = gpd.sjoin(gdf_p.copy().assign(geometry=gdf_p.centroid), area_p[['geometry']], how='inner', predicate='within').index
gdf = gdf_full.loc[idx].copy()

print(f'Manzanas detectadas en Roosevelt: {len(gdf)}')

In [ ]:
# 4. Diccionario Estandarizado y Top 5
COLS_MAP = {
    'ANALF_': 'Analfabetismo', 'BAJO_': 'Bajo logro educativo', 
    'INFANCIA_': 'Barreras primera infancia', 'INASIS_': 'Inasistencia escolar', 
    'REZAGO_': 'Rezago escolar', 'TRAB_INFAN': 'Trabajo infantil', 
    'DEPEN_': 'Dependencia económica', 'INFOR_': 'Informalidad', 
    'SALUD_': 'Barreras de salud', 'ASEGU_': 'Sin aseguramiento en salud', 
    'HACI_': 'Hacinamiento crítico', 'PARED_': 'Paredes precarias', 
    'EXCRE_': 'Eliminación inadecuada de excretas', 'PISOS_': 'Pisos precarios', 
    'AGUA_': 'Sin acceso a agua mejorada'
}

top_5 = gdf[list(COLS_MAP.keys())].mean().sort_values(ascending=False).head(5).index.tolist()
print('Top 5 variables críticas en Roosevelt:')
for v in top_5: print(f'- {COLS_MAP[v]}')

# @title Visualización: Ranking Top 5 Privaciones
top_5_means = gdf[top_5].mean().sort_values(ascending=True)
labels = [COLS_MAP[c] for c in top_5_means.index]

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, 5)) # Paleta accesible
bars = ax.barh(labels, top_5_means.values, color=colors, edgecolor='#555555', alpha=0.9)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 1, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', 
            va='center', fontsize=11, fontweight='bold', color='#2C3E50')

ax.set_title('Top 5 Variables IPM con mayor incidencia en Roosevelt', fontsize=15, fontweight='bold', pad=20)
ax.set_xlabel('Promedio de incidencia (% de hogares)', fontsize=12)
ax.set_xlim(0, top_5_means.max() + 10)
ax.grid(axis='x', linestyle='--', alpha=0.7)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Función de mapeo avanzada: Estándar Maestro Notebook 01 (Mayo 2026)
def plot_ipm_variable_advanced(gdf_zone, gdf_back, gdf_poly, column, title):
    """
    Genera cartografía profesional sincronizada con el Notebook 01 (Master):
    - Efecto Atlas: El fondo de manzanas cubre todo el lienzo.
    - Layout: Expansión del 75% a la derecha para la leyenda integrada.
    - Contraste: Regla de color de texto según valor (>0 blanco con stroke negro).
    """
    fig, ax = plt.subplots(1, 1, figsize=(18, 12))

    # --- 1. Capa Base (Fondo Maestro) ---
    # Usamos un gris muy tenue para dar contexto sin competir con la coropleta
    gdf_back.plot(ax=ax, color="#F8F8F8", edgecolor="#DCDCDC", linewidth=0.1, alpha=0.8, zorder=1)

    # --- 2. Rangos Dinámicos (5 niveles + 0) ---
    # Calculamos cortes naturales para resaltar la variabilidad interna de Roosevelt
    vals = gdf_zone[column].dropna()
    if not vals.empty and vals.max() > 0:
        breaks = sorted(list(set([0, 0.1] + list(np.linspace(vals[vals>0].min() if not vals[vals>0].empty else 1, vals.max(), 4)))))
    else:
        breaks = [0, 1, 2, 3, 4, 5]
    
    n_bins = len(breaks) - 1
    cmap = plt.get_cmap('viridis', n_bins)
    norm = BoundaryNorm(breaks, cmap.N)
    
    # --- 3. Ploteo de la Zona de Estudio ---
    gdf_zone.plot(column=column, cmap=cmap, norm=norm, edgecolor="white", linewidth=0.35, zorder=3, alpha=0.9)
    
    # Contorno destacado del área (Naranja estándar Notebook 01)
    gdf_poly.boundary.plot(ax=ax, color="#D55E00", linewidth=1.5, linestyle="-", alpha=0.8, zorder=2)

    # --- 4. Etiquetas de IPM (Lógica de Contraste Maestra) ---
    # Proyectar para centroides precisos
    centroids = gdf_zone.to_crs("EPSG:3115").geometry.centroid.to_crs(gdf_zone.crs)
    
    for x, y, label in zip(centroids.x, centroids.y, gdf_zone[column]):
        # Valor > 0: Texto BLANCO con stroke negro
        # Valor = 0: Texto NEGRO con stroke blanco
        txt_color = 'white' if label > 0 else 'black'
        stk_color = 'black' if label > 0 else 'white'
        
        ax.annotate(f'{label:.1f}%', xy=(x, y), ha='center', va='center', 
                    fontsize=8, fontweight='bold', color=txt_color,
                    path_effects=[withStroke(linewidth=1.5, foreground=stk_color, alpha=0.8)],
                    zorder=5)

    # --- 5. Ajuste de Vista 'Efecto Atlas' ---
    minx, miny, maxx, maxy = gdf_zone.total_bounds
    rx, ry = maxx - minx, maxy - miny
    
    # Aplicamos la expansión del 75% a la derecha para la leyenda (Regla Notebook 01)
    ax.set_xlim(minx - rx*0.15, maxx + rx*0.75)
    ax.set_ylim(miny - ry*0.15, maxy + ry*0.15)

    # --- 6. Leyenda Integrada ---
    legend_patches = []
    for i in range(n_bins):
        lbl = f"{breaks[i]:.1f}% - {breaks[i+1]:.1f}%"
        legend_patches.append(mpatches.Patch(facecolor=cmap(i), edgecolor='white', label=lbl))
    
    legend_patches.append(mpatches.Patch(facecolor='none', edgecolor='#D55E00', 
                                         linewidth=1.5, label='Area de intervención'))
    
    ax.legend(handles=legend_patches, 
              title=f"Privación: {title}\n(Rango IPM %)", 
              loc='center right', frameon=True, framealpha=0.9, 
              fontsize=9, title_fontsize=11)
    
    # Título Maestro
    ax.set_title(f"IPM por manzana: {title.upper()} - Roosevelt", 
                 fontsize=16, fontweight='bold', pad=20)
    
    ax.set_axis_off()
    plt.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.01)
    plt.show()

In [ ]:
# 6. Ejecución de la Cartografía Crítica (Top 5)
print("Generando mapas bajo estándar oficial 'Efecto Atlas' del Notebook 01...")
for var in top_5:
    plot_ipm_variable_advanced(gdf, gdf_fondo, gdf_area, var, COLS_MAP[var])

# Exportar GeoJSON filtrado
gdf.to_file('Mzn_ipm_variables_Roosevelt.geojson', driver='GeoJSON')

## 7. Validación de Estándares Técnicos
Este notebook cumple con las directrices de geo-informática de Santiago de Cali (Mayo 2026) y sincronización con el Notebook 01:
1. **Accesibilidad:** Uso de la paleta **Viridis** para IPM.
2. **Efecto Atlas:** Manzanas de contexto (#F8F8F8) cubriendo todo el lienzo.
3. **Contraste:** Etiquetas inteligentes (Blanco/Negro) con stroke de 1.5.
4. **Layout:** Expansión del 75% a la derecha para leyenda integrada.
5. **Color de Borde:** Naranja estándar (#D55E00) para el área de estudio.
6. **Precisión:** Datos ajustados a **1 decimal**.

In [ ]:
# 8. Descargar GeoJSON (Colab)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('Mzn_ipm_variables_Roosevelt.geojson')
    print('Descarga iniciada: Mzn_ipm_variables_Roosevelt.geojson')
else:
    print(f'Archivo disponible localmente.')